# <center>Clustering Analysis<center>

<p>Team Name: Team Regular
<p>Student Names: Alameen Adeku, Adam Rodi, Adriean Lemoine, Nicholas Burgo

## Instructions
Use generic coding style unless hard-coded values are really necessary.<br>
Your code must be efficient and use self-explanatory naming.<br>
Use appropriate Python library methods for each task instead of using loops.<br>
Run your entire code and save. Then submit this <b>saved</b> copy.

## Imports

In [30]:
import numpy as np
import os
import pandas as pd
from pathlib import Path
from scipy import ndimage
from skimage.filters.rank import entropy
from skimage.morphology import ball
from sklearn.preprocessing import MinMaxScaler
import tifffile as tfl

## Configurations

In [31]:
# --- Directory Configs ---

data_dir = os.path.join('..', 'Data')
input_data_dir = os.path.join(data_dir, 'input/zebrafish_tiff_scans')
output_dir = os.path.join(data_dir, 'output')
output_csv_dir = os.path.join(output_dir, 'csv')
output_figures_dir = os.path.join(output_data_dir, 'figures')

# Format of data files which prefix the sequential sample number
# Ex: 'ZS-1.tif' would have a data file prefix of 'ZS-'
data_file_prefix = 'ZS-' 

groups = {
    "control": "control",
    "anesthetic": "anesthetic",
    "stimulant": "stimulant"
}


## Functions

In [32]:
def load_tiff_stack(path):
    '''Load a single TIFF file into a 3D array'''
    sample = tfl.imread(path).astype(float)
    return sample

def compute_sample_gradient(tiff_stack):
    """Compute a 3D gradient magnitude map using Sobel filters along all axes."""
    gx = ndimage.sobel(tiff_stack, axis=0)
    gy = ndimage.sobel(tiff_stack, axis=1)
    gz = ndimage.sobel(tiff_stack, axis=2)
    sample_gradients = np.sqrt(gx**2 + gy**2 + gz**2)
    # print("Gradient sample (first 10 values):", np.round(sample_gradients.ravel()[:10], 3))
    return sample_gradients

def compute_sample_texture(tiff_stack, size=3):
    """Compute a 3D texture map as the local standard deviation of voxel intensities in a cubic neighborhood."""
    if isinstance(size, int):
        size = (size, size, size)
    
    # local mean of X
    mean = ndimage.uniform_filter(tiff_stack, size=size, mode='reflect')
    # local mean of X^2
    mean_sq = ndimage.uniform_filter(tiff_stack * tiff_stack, size=size, mode='reflect')
    
    # variance = E[X^2] - (E[X])^2
    var = mean_sq - mean * mean
    var = np.maximum(var, 0.0)  # avoid small negative due to rounding
    sample_textures = np.sqrt(var)
    # print("Texture sample (first 10 values):", np.round(sample_textures.ravel()[:10], 3))
    return sample_textures

def compute_sample_average(array_3d):
    """Compute the mean (average) of all values in a 3D array."""
    sample_average = float(np.mean(array_3d))  
    return sample_average

def compute_sample_std(array_3d):
    """Compute the standard deviation of all values in a 3D array."""
    sample_std = float(np.std(array_3d))
    return sample_std
    
def summarize_fish_sample(n, fish_class, intensity_array, gradient_array, texture_array):
    """Consolidate all summary data for a single fish sample into a dictionary of values."""
    return {
        "fish_id": n,
        "fish_class": fish_class,
        "avg_intensity": compute_sample_average(intensity_array),
        "avg_gradient": compute_sample_average(gradient_array),
        "avg_texture": compute_sample_average(texture_array),
        "std_intensity": compute_sample_std(intensity_array),
        "std_gradient": compute_sample_std(gradient_array),
        "std_texture": compute_sample_std(texture_array)
    }

def get_fish_sample(n, fish_class, path):
    """Takes in fish sample metadata, calculates intensity/gradient/texture data, and outputs a single dictionary summarizing the fish data."""
    intensity_array = load_tiff_stack(path)
    gradient_array = compute_sample_gradient(intensity_array)
    texture_array = compute_sample_texture(intensity_array)
    summary_dict = summarize_fish_sample(n, fish_class, intensity_array, gradient_array, texture_array)
    return summary_dict

def build_dataset(input_data_dir, groups, file_prefix="ZS-", num_files=10):
    """
    Process all zebrafish TIFF scans across groups and return a list of per-fish summary dictionaries.
    
    Args:
        input_data_dir : str
            Path to the folder containing group subfolders.
        groups : dict
            Dictionary mapping group names (str) to subfolder names.
            Example: {"control": "control", "anesthetic": "anesthetic", "stimulant": "stimulant"}
        file_prefix : str
            Prefix for fish filenames (default "ZS-")
        num_files : int
            Number of fish per group (default 10)
    
    Returns:
        samples : list of dict
            Each dict contains fish_id, fish_class, avg/std of intensity, gradient, and texture.
    """
    dataset = []
    fish_id = 1

    for fish_class, group_folder in groups.items():
        group_path = Path(input_data_dir) / group_folder

        for i in range(1, num_files + 1):
            filename = f"{file_prefix}{i}.tif"
            file_path = group_path / filename
            
            if not file_path.exists():
                print(f"Warning: File not found: {file_path}")
                continue

            # Compute single-fish sample dictionary
            sample = get_fish_sample(fish_id, fish_class, str(file_path))
            dataset.append(sample)
            fish_id += 1
    
    return dataset

def dataset_to_dataframe(dataset):
    """Convert a list of per-fish dictionaries into a pandas DataFrame."""
    df = pd.DataFrame(dataset)
    return df

def dataframe_to_csv(df, output_dir, filename="dataset.csv"):
    """Save a pandas DataFrame to a CSV file in the specified directory."""
    os.makedirs(output_dir, exist_ok=True)  # make sure directory exists
    csv_path = os.path.join(output_dir, filename)
    df.to_csv(csv_path, index=False)        # index=False to omit row numbers
    print(f"DataFrame saved to: {csv_path}")
    return csv_path

def normalize_dataframe(df, feature_columns):
    """
    Normalize specified columns of a DataFrame to the 0-1 range using Min-Max scaling.
    
    Args:
        df : pd.DataFrame
            The input DataFrame.
        feature_columns : list of str
            List of column names to normalize 
            Example: numeric feature columns
    
    Returns:
        df_normalized : pd.DataFrame
            A copy of the DataFrame with the specified columns normalized to [0, 1].
    """
    df_normalized = df.copy()
    scaler = MinMaxScaler()
    
    # Fit and transform only the selected feature columns
    df_normalized[feature_columns] = scaler.fit_transform(df_normalized[feature_columns])
    
    return df_normalized



## Read Data

### Load Fish Scans

In [27]:
# Build the dataset
dataset = build_dataset(input_data_dir, groups, file_prefix="ZS-", num_files=10)

# Convert to DataFrame
df = dataset_to_dataframe(dataset)

# Output dataframe to csv
csv_path = dataframe_to_csv(df, output_csv_dir)


DataFrame saved to: ../Data/output/csv/dataset.csv


In [23]:
# Inspect
print(df)


    fish_id  fish_class  avg_intensity  avg_gradient  avg_texture  \
0         1     control      97.159876    346.923257    15.667902   
1         2     control      98.905908    346.061106    15.470677   
2         3     control     124.755537    502.162567    20.565100   
3         4     control     147.088776    605.174318    23.872697   
4         5     control      60.005725    145.230265     8.693171   
5         6     control      76.774804    233.951149    11.528386   
6         7     control     106.289575    385.292421    17.218341   
7         8     control      75.176766    225.752835    11.634854   
8         9     control     139.302376    584.807681    22.825506   
9        10     control      99.290657    350.204488    15.864276   
10       11  anesthetic      85.001966    264.351582    13.091270   
11       12  anesthetic      65.952087    155.670189     9.056834   
12       13  anesthetic      58.984134    122.977657     7.604496   
13       14  anesthetic      71.04

In [33]:
# Normalize dataframe
feature_cols = ['avg_intensity', 'avg_gradient', 'avg_texture', 'std_intensity', 'std_gradient', 'std_texture']
df_norm = normalize_dataframe(df, feature_cols)

In [34]:
print(df_norm)

    fish_id  fish_class  avg_intensity  avg_gradient  avg_texture  \
0         1     control       0.236817      0.298800     0.326771   
1         2     control       0.247423      0.297686     0.319110   
2         3     control       0.404437      0.499440     0.517003   
3         4     control       0.540093      0.632578     0.645487   
4         5     control       0.011138      0.038122     0.055837   
5         6     control       0.112995      0.152789     0.165971   
6         7     control       0.292272      0.348391     0.386998   
7         8     control       0.103289      0.142193     0.170107   
8         9     control       0.492797      0.606255     0.604809   
9        10     control       0.249760      0.303041     0.334399   
10       11  anesthetic       0.162968      0.192080     0.226681   
11       12  anesthetic       0.047257      0.051615     0.069963   
12       13  anesthetic       0.004932      0.009361     0.013547   
13       14  anesthetic       0.07

## Visual Exploration of Data

### Histograms

### Distributions

### Box-Whisker Plots

### Violin Plots

## Data Quality & Cleaning

Instruction: Add a comment for each method

## Handling Redundancy

### X-square Test

### Correlation Analysis

### Visual Exploration (scatter-plot matrix)

## Dimensionality Reduction

### PCA

## Discretization

### Histogram of Discretized Attribute

### X-square Test of Discretized Attributes

### Visual Exploration (scatter-plot matrix) of Discretized Attributes

## Feature Selection/Generation

### Select Features

### Generate Features

# Generate Clusters

## K-means

## Hierarchical

# Evaluation of Clusters

See instructions provided in the report template

## <center> REFERENCES </center>
List resources (book, internet page, etc.) that you used to complete this challenge.

- https://numpy.org/doc/2.3/index.html
- https://pandas.pydata.org/docs/
- http://pypi.org/project/tifffile/